# TabFM Financial Risk Benchmark -- Colab runner

Runs the parts of [`tabfm-financial-risk-benchmark`](https://github.com/ConceptualCode/tabfm-financial-risk-benchmark) that need more RAM/GPU than a typical laptop has:

- **TabFM**: ~6.5GB checkpoint, needs ~12-16GB RAM to load (its loader stages through CPU RAM before moving to GPU, so a small GPU alone doesn't fix a small-RAM machine -- see `models.py::build_tabfm`).
- **SAP-RPT** (`sap-rpt-1-oss`): tiny weights (~65MB) but its default settings (`bagging=8`, `max_context_size=8192`) can need up to ~80GB of GPU memory for inference-time activations. We constrain both via env vars below to fit a free-tier T4 (16GB VRAM).

**Before running:** go to Runtime -> Change runtime type -> select a GPU (T4 is fine, free tier).

**HF access:** `sap_rpt` requires requesting access at https://huggingface.co/SAP/sap-rpt-1-oss first. If you don't have access yet, skip the `sap_rpt` entries in the run cells below -- everything else still works.

## 1. Sanity-check the runtime (GPU + RAM)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h

## 2. Clone the repo and install dependencies

Colab ships Python 3.11/3.12 already, so no Python-version workaround is needed here (unlike the laptop issue we hit locally, which required upgrading from 3.10).

In [ ]:
!python --version
!git clone https://github.com/ConceptualCode/tabfm-financial-risk-benchmark.git
%cd tabfm-financial-risk-benchmark

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. HuggingFace auth (needed only for `sap_rpt`)

Add your token via Colab's Secrets manager (key icon in the left sidebar) as `HF_TOKEN` -- do **not** paste it directly into a cell, since that embeds it in the notebook file if you ever save/share it.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')  # requires the HF_TOKEN secret to be set, see markdown above
login(token=hf_token)

## 4. Constrain SAP-RPT to fit the T4's 16GB VRAM

Default settings (`bagging=8`, `max_context_size=8192`) target ~80GB GPUs. These reduced values are a starting point -- push them up if you have headroom, or down further if you still hit an OOM.

In [ ]:
import os
os.environ['SAP_RPT_BAGGING'] = '2'
os.environ['SAP_RPT_MAX_CONTEXT_SIZE'] = '1024'

## 5. Smoke tests -- confirm the base setup works before touching the foundation models

In [ ]:
!python -m pytest tests/test_smoke.py -v

## 6. Validation run -- one dataset, skip SHAP

Confirms TabFM and SAP-RPT actually load and predict end to end before committing to the full grid. Drop `sap_rpt` from `--models` if you don't have HF access yet.

In [ ]:
!python scripts/run_benchmark.py --datasets cd1 --models tabfm sap_rpt xgboost --skip-shap

## 7. Full benchmark

All 10 FinBench datasets, all models, including SHAP. This is the long-running step -- Colab free-tier sessions can disconnect after ~12 hours idle/max runtime, so if this is going to run long, consider chunking `--datasets` across a few cells/sessions and concatenating the resulting CSVs afterward.

In [ ]:
!python scripts/run_benchmark.py

## 8. Pull the results back down

Zips `results/` and downloads it, so you can drop it into your local clone of the repo (merging with anything you ran locally for the GBM baselines) and commit from there.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('results', 'zip', 'results')
files.download('results.zip')